# SEED-VII EEGNet × LoRA-LLM  —  算力平台镜像 Pipeline  (v4)

**适用场景**: 租用 GPU 算力平台 (AutoDL / 矩池云 / 恒源云 等)，
从零拉取数据 → 预处理 → 训练，全部自动化，**无需交互**。

| 步骤 | 说明 |
|------|------|
| 1. 配置 | 修改 Cell 2 中的路径和超参 |
| 2. 环境 | git clone + pip install（分步显示进度，不会假死）|
| 3. 数据 | ModelScope 拉取 `DEREKVERSE/SEED-VII` |
| 4. LLM  | ModelScope 拉取 `Qwen/Qwen2.5-0.5B-Instruct` |
| 5. NPZ   | 预处理 4s 滑动窗口 → shard NPZ |
| 6. 训练  | EEGNet + LoRA-LLM + **MoCo 动量队列 (K=4096)** |
| 7. 继续  | 断点续训；验证集宏 F1 最优自动保存 best.pt |


## 0. 用户配置区 (修改这里即可)

In [ ]:
import os, sys
from pathlib import Path

# ========== 必填 ==========
MODELSCOPE_TOKEN = os.environ.get('MODELSCOPE_TOKEN', '')  # 私有数据集才需要
DATASET_ID       = 'DEREKVERSE/SEED-VII'
LLM_MODEL_ID     = 'Qwen/Qwen2.5-0.5B-Instruct'

# ========== 路径 (按你的算力平台调整) ==========
WORK    = Path('/mnt/workspace')   # 持久化目录
REPO    = WORK / 'EEG_OPUS1'       # clone 到这里
MODEL_DIR    = WORK / 'models' / 'Qwen2.5-0.5B-Instruct'
DATASET_DIR  = WORK / 'seedvii_ms_dataset'
NPZ_DIR      = WORK / 'seedvii_npz'
RUN_DIR      = WORK / 'seedvii_contrastive_runs' / 'run_valence3'

# ========== 训练超参 ==========
TRAIN_BATCH_SIZE  = 96
TRAIN_EPOCHS      = 50
QUEUE_SIZE        = 4096      # MoCo 队列大小

for d in [WORK, MODEL_DIR.parent, DATASET_DIR, NPZ_DIR, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('WORK:', WORK)
print('Python:', sys.version)
print('GPU:', os.popen('nvidia-smi -L 2>/dev/null || echo N/A').read().strip())

## 1. 环境安装

In [ ]:
# ── Step 1/4: git clone (fast, skips if exists) ──
print("[1/4] Checking repo ...", flush=True)
if not (REPO / "seedvii_modal_contrastive_lora" / "pyproject.toml").exists():
    !rm -rf {REPO} 2>/dev/null
    !git clone https://github.com/PRIMOCOSMOS/EEG_OPUS1.git {REPO}
else:
    print("  [OK] Repo already exists")

PROJ = REPO / "seedvii_modal_contrastive_lora"

# ── Step 2/4: install PyTorch FIRST (biggest package, can take 5-15 min) ──
print("[2/4] Installing PyTorch (this may take several minutes) ...", flush=True)
!pip install torch
print("  [OK] PyTorch installed")

# ── Step 3/4: install remaining dependencies ──
print("[3/4] Installing other packages ...", flush=True)
!pip install transformers peft modelscope h5py pyyaml tqdm scipy pandas
print("  [OK] Dependencies installed")

# ── Step 4/4: install this library in editable mode ──
print("[4/4] Installing seedvii_contrastive (editable) ...", flush=True)
!pip install -e {PROJ}

import sys, os
# Robust path fix (works even if editable install is flaky)
if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))
if str(PROJ / "seedvii_contrastive") not in sys.path:
    sys.path.insert(0, str(PROJ / "seedvii_contrastive"))
os.chdir(str(PROJ))

# ── verify ──
import torch
print(f"[OK] Environment ready  |  torch={torch.__version__}  cuda={torch.cuda.is_available()}")


## 2. 拉取 SEED-VII 数据集 (ModelScope)

In [ ]:
from seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths

# 检测是否已下载
EEG_ROOT, TEXT_CSV = find_downloaded_paths(DATASET_DIR)

if EEG_ROOT is None or TEXT_CSV is None:
    print('[Download] Pulling from ModelScope ...')
    !python -m seedvii_contrastive.scripts.download_modelscope_seedvii \
      --dataset-id {DATASET_ID} \
      --local-dir {DATASET_DIR} \
      --max-workers 4
    EEG_ROOT, TEXT_CSV = find_downloaded_paths(DATASET_DIR)

assert EEG_ROOT is not None, '1-20.mat not found'
assert TEXT_CSV is not None, 'text_protocol.csv not found'
print(f'[OK] EEG_ROOT={EEG_ROOT}')
print(f'[OK] TEXT_CSV={TEXT_CSV}')

## 3. 拉取 LLM (Qwen2.5-0.5B-Instruct)

In [ ]:
from modelscope import snapshot_download

if not (MODEL_DIR / 'config.json').exists():
    print('[LLM] Downloading ...')
    mp = snapshot_download(LLM_MODEL_ID, cache_dir=str(MODEL_DIR.parent))
    mp = Path(mp)
    if mp.resolve() != MODEL_DIR.resolve():
        import shutil
        if MODEL_DIR.exists(): shutil.rmtree(MODEL_DIR)
        shutil.copytree(str(mp), str(MODEL_DIR), symlinks=True)
    print(f'[LLM] Downloaded to {MODEL_DIR}')
else:
    print(f'[LLM] Already at {MODEL_DIR}')

## 4. NPZ 预处理 (4s 滑动窗口)

In [ ]:
# ── 4.0 (可选) 把经典 .mat 转成真正的 HDF5(v7.3) 以便 h5py 懒加载 ──
# 你的 .mat 是经典 v5 格式、字段名是干净的 '1'..'80'，内存 50GB 足够，
# 其实可以【直接预处理经典 .mat】，不必转换。这里默认【跳过转换】。
#
# 若你确实想用懒加载(例如内存很小)，把 DO_CONVERT 设为 True。
# 注意：本 cell 使用【已修复】的 h5py 转换器(原版 savemat(format="7.3") 必失败)。
DO_CONVERT = False   # 默认 False：直接用经典 .mat 预处理（推荐，你内存够）

if DO_CONVERT:
    CONVERTED_DIR = WORK / 'seedvii_hdf5'
    CONVERTED_DIR.mkdir(parents=True, exist_ok=True)

    def _is_valid_hdf5(p):
        try:
            return p.stat().st_size > 1024 and open(p, 'rb').read(8) == b'\x89HDF\r\n\x1a\n'
        except OSError:
            return False

    need = not all(_is_valid_hdf5(CONVERTED_DIR / f'{s}.mat') for s in range(1, 21))
    if need:
        print('[Convert] 经典 .mat -> HDF5 (已修复的 h5py 转换器)...')
        !PYTHONPATH={PROJ} python -m seedvii_contrastive.scripts.convert_classic_mat_to_hdf5 \
          --input-dir {EEG_ROOT} \
          --output-dir {CONVERTED_DIR} \
          --subjects 1-20 \
          --key-mode numeric
    else:
        print('[Convert] 已存在有效 HDF5，跳过。')
    EEG_ROOT = CONVERTED_DIR
    print(f'[OK] 使用懒加载 EEG_ROOT = {EEG_ROOT}')
else:
    print(f'[Convert] 跳过转换，直接预处理经典 .mat。EEG_ROOT = {EEG_ROOT}')


In [ ]:
# ── 4.1 NPZ 预处理（进程内运行：实时进度 + 真实报错，不再"进度条后假死"）──
# 用 !python 子进程会块缓冲 stdout，导致进度条出现后看似卡死、报错被吞。
# 这里直接在内核进程内调用函数：任何异常都会立刻抛出完整 traceback。
assert 'EEG_ROOT' in dir() and EEG_ROOT is not None, "EEG_ROOT 未设置 — 先跑 Cell 6"

import sys
if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))

from seedvii_contrastive.data.preprocess import preprocess_to_npz

if not (NPZ_DIR / 'index.csv').exists():
    print('[NPZ] 进程内预处理 20 被试 × 80 trial ...', flush=True)
    print('[NPZ] 经典 .mat 会逐个 loadmat 进内存(约1-2GB/人)，50GB 内存足够。', flush=True)
    index = preprocess_to_npz(
        output_dir=str(NPZ_DIR),
        input_root=str(EEG_ROOT),
        subjects=list(range(1, 21)),
        fs=200,
        window_sec=4.0,
        stride_sec=4.0,
        center_ratio=0.60,
        max_windows_per_clip=12,
        shard_size=512,
    )
    n = len(list(NPZ_DIR.glob('shard_*.npz')))
    print(f'[NPZ] 完成 — index={index}，共 {n} 个 shard', flush=True)
else:
    print(f'[NPZ] 已存在 {NPZ_DIR / "index.csv"}，跳过', flush=True)


## 5. 写入运行时配置

In [ ]:
import yaml

base_cfg_path = PROJ / 'configs' / 'modelscope_default.yaml'
cfg = yaml.safe_load(open(base_cfg_path, 'r', encoding='utf-8'))

# 覆写路径
cfg['data'].update({
    'modelscope_dataset_id': DATASET_ID,
    'local_dataset_dir': str(DATASET_DIR),
    'eeg_root': str(EEG_ROOT),
    'text_csv_path': str(TEXT_CSV),
    'npz_dir': str(NPZ_DIR),
})
cfg['runtime']['output_dir'] = str(RUN_DIR)
cfg['model']['llm']['model_name_or_path'] = str(MODEL_DIR)
cfg['train']['batch_size'] = TRAIN_BATCH_SIZE
cfg['train']['epochs'] = TRAIN_EPOCHS
cfg['moco']['queue_size'] = QUEUE_SIZE

RUN_DIR.mkdir(parents=True, exist_ok=True)
run_cfg = RUN_DIR / 'config.yaml'
yaml.safe_dump(cfg, open(run_cfg, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print(open(run_cfg, 'r', encoding='utf-8').read())

## 6. 训练  (MoCo v3)

In [ ]:
# 前台训练：自动检测 last.pt 续训
!python -m seedvii_contrastive.scripts.train_contrastive --config {run_cfg}

## 7. 推理 / 导出 Embedding

In [ ]:
BEST = RUN_DIR / 'best.pt'
OUT_EMB = RUN_DIR / 'val_embeddings.npz'

if BEST.exists():
    !python -m seedvii_contrastive.scripts.encode_eeg \
      --config {run_cfg} \
      --checkpoint {BEST} \
      --split val \
      --out {OUT_EMB}
    print('saved:', OUT_EMB)
else:
    print('[WARN] best.pt not found — train first')